# 03 · Train the cloak generator

Run the smoke test first (300 steps, ~20–30 min). It should show the loss
falling and `lpips_raw` settling near the 0.05 budget, and the sample grids
(clean / cloaked / |δ|×10) should look clean in the middle row.

The main run checkpoints to Drive every 500 steps and stops itself before
the session limit (`budget_min`). **If Colab disconnects, rerun this notebook
top to bottom**: it resumes from `last.pt` automatically.

In [ ]:
# ---- edit these ----
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"   # "" = copy from Drive
BRANCH     = "master"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # runs, data tar and (optionally) code live here
# Private repo: add a Colab secret GITHUB_TOKEN. FLUX VAE: add HF_TOKEN.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
CODE = "/content/Project-Soteria"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if REPO_URL:
    url = REPO_URL
    if secret("GITHUB_TOKEN"):
        url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
    if Path(CODE).exists():
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, url, CODE], check=True)
else:
    shutil.copytree(f"{DRIVE_ROOT}/code/Project-Soteria", CODE, dirs_exist_ok=True)

if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.chdir(CODE)
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Unpack the prepared dataset from Drive onto local disk (much faster to read).
if not Path("/content/data/prepared/manifest.csv").exists():
    Path("/content/data").mkdir(exist_ok=True)
    !tar -xf {DRIVE_ROOT}/data/prepared.tar -C /content/data
!head -3 /content/data/prepared/manifest.csv && wc -l /content/data/prepared/manifest.csv

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_ROOT}/runs

In [ ]:
!python -m soteria.train --config configs/smoke.yaml out_dir={DRIVE_ROOT}/runs/smoke

In [ ]:
import glob
from IPython.display import Image as Show
Show(sorted(glob.glob(f"{DRIVE_ROOT}/runs/smoke/samples/*.png"))[-1])

In [ ]:
# Main run. Safe to re-execute after a disconnect: it resumes from last.pt.
!python -m soteria.train --config configs/base_256.yaml out_dir={DRIVE_ROOT}/runs/base_256